In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

dataset = load_breast_cancer()
X = dataset.data
y = dataset.target

print("Features shape:", X.shape)
print("Class names:", dataset.target_names) 

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

baseline = LogisticRegression(
    max_iter=2000,
    random_state=42
)
baseline.fit(X_train_scaled, y_train)

y_pred = baseline.predict(X_test_scaled)
baseline_accuracy = accuracy_score(y_test, y_pred)

print("Training samples:", len(y_train))
print("Test samples:", len(y_test))
print(f"Baseline test accuracy: {baseline_accuracy:.4f}")
print(f"Correct predictions: {(y_pred == y_test).sum()}/{len(y_test)}")

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader

X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)
y_train_tensor = torch.tensor(
    y_train, dtype=torch.float32
).reshape(-1, 1)
y_test_tensor = torch.tensor(
    y_test, dtype=torch.float32
).reshape(-1, 1)

classification_dataset = TensorDataset(
    X_train_tensor, y_train_tensor
)


def train_classifier(link, epochs=500, initial_lr=0.001):
    torch.manual_seed(42)
    model = nn.Linear(X_train_tensor.shape[1], 1)

    loader = DataLoader(
        classification_dataset,
        batch_size=1,
        shuffle=True,
        generator=torch.Generator().manual_seed(42)
    )

    optimizer = torch.optim.SGD(
        model.parameters(), lr=initial_lr
    )
    normal = torch.distributions.Normal(0.0, 1.0)

    def compute_loss(z, targets):
        if link == "logistic":
            return F.binary_cross_entropy_with_logits(z, targets)

        if link == "probit":
            probabilities = normal.cdf(z).clamp(1e-7, 1 - 1e-7)
            return F.binary_cross_entropy(probabilities, targets)

        raise ValueError("Unknown link")

    history = []
    model.train()

    for epoch in range(epochs):
        lr = initial_lr / (1 + epoch / 100)
        for group in optimizer.param_groups:
            group["lr"] = lr

        for features, target in loader:
            optimizer.zero_grad()
            loss = compute_loss(model(features), target)
            loss.backward()
            optimizer.step()

        with torch.no_grad():
            train_loss = compute_loss(
                model(X_train_tensor), y_train_tensor
            ).item()

        history.append(train_loss)

        if (epoch + 1) % 100 == 0:
            print(
                f"{link}, epoch {epoch + 1}: "
                f"training loss = {train_loss:.6f}"
            )

    model.eval()
    with torch.no_grad():
        z = model(X_test_tensor)
        probabilities = (
            torch.sigmoid(z) if link == "logistic"
            else normal.cdf(z)
        )
        predictions = (probabilities >= 0.5).float()
        accuracy = (
            predictions == y_test_tensor
        ).float().mean().item()

    return model, history, probabilities, predictions, accuracy


(
    model_logistic,
    loss_history,
    test_probabilities,
    test_predictions,
    logistic_accuracy
) = train_classifier("logistic")

(
    model_probit,
    probit_loss_history,
    probit_probabilities,
    probit_predictions,
    probit_accuracy
) = train_classifier("probit")

print(f"sklearn baseline accuracy: {baseline_accuracy:.4f}")
print(f"PyTorch logistic accuracy: {logistic_accuracy:.4f}")
print(f"PyTorch probit accuracy:   {probit_accuracy:.4f}")

print(
    "Number of different predictions:",
    (test_predictions != probit_predictions).sum().item()
)

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

ale_df = pd.read_csv("mcs_ds_edited_iter_shuffled.csv"
)

X_reg = ale_df.iloc[:, :4]
y_reg = ale_df["ale"]

X_reg_train, X_reg_test, y_reg_train, y_reg_test = (
    train_test_split(
        X_reg,
        y_reg,
        test_size=0.2,
        random_state=42
    )
)

reg_scaler = StandardScaler()
X_reg_train_scaled = reg_scaler.fit_transform(X_reg_train)
X_reg_test_scaled = reg_scaler.transform(X_reg_test)

reg_baseline = LinearRegression()
reg_baseline.fit(X_reg_train_scaled, y_reg_train)

baseline_reg_predictions = reg_baseline.predict(
    X_reg_test_scaled
)
baseline_mae = mean_absolute_error(
    y_reg_test, baseline_reg_predictions
)

print(f"Baseline test MAE: {baseline_mae:.6f}")

In [ ]:
X_reg_train_tensor = torch.tensor(
    X_reg_train_scaled, dtype=torch.float64
)
X_reg_test_tensor = torch.tensor(
    X_reg_test_scaled, dtype=torch.float64
)
y_reg_train_tensor = torch.tensor(
    y_reg_train.to_numpy(), dtype=torch.float64
).reshape(-1, 1)

reg_dataset = TensorDataset(
    X_reg_train_tensor, y_reg_train_tensor
)


def train_regression(loss_type, epochs=3000, initial_lr=0.001):
    torch.manual_seed(42)
    model = nn.Linear(
        X_reg_train_tensor.shape[1], 1
    ).double()

    generator = torch.Generator().manual_seed(42)
    loader = DataLoader(
        reg_dataset,
        batch_size=1,
        shuffle=True,
        generator=generator
    )

    optimizer = torch.optim.SGD(
        model.parameters(),
        lr=initial_lr
    )

    student_t = torch.distributions.StudentT(
        df=torch.tensor(5.0, dtype=torch.float64)
    )

    def compute_loss(predictions, targets):
        residuals = targets - predictions

        if loss_type == "mse":
            return residuals.square().mean()

        if loss_type == "student_t":
            return -student_t.log_prob(residuals).mean()

        raise ValueError("Unknown loss type")

    history = []
    model.train()

    for epoch in range(epochs):
        lr = initial_lr / (1 + epoch / 500)
        for group in optimizer.param_groups:
            group["lr"] = lr

        for features, target in loader:
            optimizer.zero_grad()

            prediction = model(features)
            loss = compute_loss(prediction, target)

            loss.backward()
            optimizer.step()

        with torch.no_grad():
            train_loss = compute_loss(
                model(X_reg_train_tensor),
                y_reg_train_tensor
            ).item()

        history.append(train_loss)

        if (epoch + 1) % 500 == 0:
            print(
                f"{loss_type}, epoch {epoch + 1}: "
                f"training loss = {train_loss:.6f}"
            )

    model.eval()
    with torch.no_grad():
        test_predictions = (
            model(X_reg_test_tensor)
            .squeeze(1)
            .numpy()
        )

    test_mae = mean_absolute_error(
        y_reg_test, test_predictions
    )

    return model, history, test_predictions, test_mae


model_mse, history_mse, predictions_mse, mse_mae = (
    train_regression("mse")
)

model_t, history_t, predictions_t, t_mae = (
    train_regression("student_t")
)

In [ ]:
reg_results = pd.DataFrame({
    "Model": [
        "sklearn Linear Regression",
        "PyTorch Linear Regression (MSE + SGD)",
        "PyTorch Linear Regression (Student-t + SGD)"
    ],
    "Test MAE": [
        baseline_mae,
        mse_mae,
        t_mae
    ]
})

display(reg_results)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

for ax, history, title in zip(
    axes,
    [history_mse, history_t],
    ["MSE training loss", "Student-t training loss"]
):
    n = min(200, len(history))
    ax.plot(range(1, n + 1), history[:n])
    ax.set_xlim(0, 200)
    ax.set_title(title)
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Training loss")
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()